![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Processing and Database Design

*Project 1 — SQL: From Data to Insight*

**Team:**
**Dataset:**

---

Turning one flat file into a relational database: clean the data, split it into tables, wire up the keys, load it into SQLite.

> **Done when:** `data/project.db` exists with 3+ related tables, `sql/schema.sql` matches your ERD, no foreign key is dangling, and the row counts are what you expect.

The sections below are a suggested route, not a form to fill in. Add sections, drop them, reorder them — what is graded is the analysis, not whether you kept the scaffold.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

In [2]:
import sqlite3

---
## 1. Load the raw data

In [3]:
RAW = "../data/raw/"

production = pd.read_csv(RAW + "FAOSTAT_data_en_9-25-2026.csv",     encoding="utf-8-sig")
trade      = pd.read_csv(RAW + "FAOSTAT_data_en_9-25-2026 (1).csv", encoding="utf-8-sig")
processed  = pd.read_csv(RAW + "FAOSTAT_data_en_9-25-2026 (2).csv", encoding="utf-8-sig")

print(production.shape, trade.shape, processed.shape)

(15622, 15) (40306, 15) (1421, 15)


---
## 2. Clean

Work through the problems you listed in notebook 01. Every decision you make here — drop the nulls or keep them, cast or discard — is one you may be asked to defend on Friday, so note why.

In [4]:
CLEAN = "../data/clean/"

production_clean = pd.read_csv(CLEAN + "production_clean.csv")
trade_clean      = pd.read_csv(CLEAN + "trade_clean.csv")

print(production_clean.shape, trade_clean.shape)

(8082, 9) (33097, 9)


**Cleaning decisions**
- Kept only rows flagged **A** (official values) — everything else (estimated, imputed, missing) was dropped, done in `clean_data.py`.
- Countries were left exactly as FAOSTAT reports them, including aggregates like "China" (mainland + Taiwan + Hong Kong + Macao) — no rows removed on that basis.
- Kept only the columns needed for the database, renamed to short lowercase names for SQL.
- Kept zero values: 0 t is real information. Tiny quantities are filtered in the SQL queries when calculating price per kg.

---
## 3. Design your tables

Decide what your tables are and how they relate before you write any code. Draw the ERD (Excalidraw, draw.io), export it as an image, commit it, and link it from your README.

Three tables minimum, with real primary and foreign keys.

**Database design: 3 lookup tables + 2 fact tables**

Lookup tables (parents):
- `countries` (**area_code** PK, area_name)
- `elements` (**element_code** PK, element_name, unit)
- `flags` (**flag** PK, flag_description)

Fact tables (children). Each raw file stays its own table:
- `production` (area_code FK, year, element_code FK, value, flag FK)
- `trade` (area_code FK, year, element_code FK, value, flag FK)

*(`processed` was dropped: under flag A only it had just 2 rows, too sparse to analyze.)*

---
## 4. Build the tables

Split the cleaned data into the tables you designed.

In [5]:
# Lookup tables: list every country / element / flag that appears in any of the 3 files
countries = pd.concat([
    production_clean[["area_code", "area_name"]],
    trade_clean[["area_code", "area_name"]],
]).drop_duplicates().sort_values("area_code").reset_index(drop=True)

elements = pd.concat([
    production_clean[["element_code", "element_name", "unit"]],
    trade_clean[["element_code", "element_name", "unit"]],
]).drop_duplicates().sort_values("element_code").reset_index(drop=True)

flags = pd.concat([
    production_clean[["flag", "flag_description"]],
    trade_clean[["flag", "flag_description"]],
]).drop_duplicates().sort_values("flag").reset_index(drop=True)

# Fact tables: keep only the key columns + value + flag
fact_cols = ["area_code", "year", "element_code", "value", "flag"]
production_table = production_clean[fact_cols]
trade_table      = trade_clean[fact_cols]

print("countries:", len(countries), "| elements:", len(elements), "| flags:", len(flags))
display(elements, flags)

countries: 203 | elements: 7 | flags: 1


,element_code,element_name,unit
0,5312,Area harvested,ha
1,5412,Yield,kg/ha
2,5510,Production,t
3,5610,Import quantity,t
4,5622,Import value,1000 USD
5,5910,Export quantity,t
6,5922,Export value,1000 USD


,flag,flag_description
0,A,Official value


---
## 5. Check the foreign keys

**Do not skip this.** A dangling foreign key is the failure that costs people their Wednesday: the database loads without an error, then your joins silently drop rows and every number is quietly wrong. Check every key value exists in its parent table before you load anything.

In [6]:
for name, table in {"production": production_table, "trade": trade_table}.items():
    bad_country = (~table["area_code"].isin(countries["area_code"])).sum()
    bad_element = (~table["element_code"].isin(elements["element_code"])).sum()
    bad_flag    = (~table["flag"].isin(flags["flag"])).sum()
    dup_keys    = table.duplicated(["area_code", "year", "element_code"]).sum()
    print(f"{name}: missing countries={bad_country}, missing elements={bad_element}, "
          f"missing flags={bad_flag}, duplicate keys={dup_keys}")

production: missing countries=0, missing elements=0, missing flags=0, duplicate keys=0
trade: missing countries=0, missing elements=0, missing flags=0, duplicate keys=0


All foreign keys exist in their parent tables and no primary key is duplicated, so the tables are safe to load.

---
## 6. Create the database and load it

Write your `CREATE TABLE` statements in `sql/schema.sql`, then load the tables — parents before children, or the foreign keys will be rejected.

In [7]:
schema_statements = open("../sql/schema.sql").read().split(";")
schema_statements = [s.strip() for s in schema_statements if s.strip()]

In [8]:
conn = sqlite3.connect("../data/project.db")       # creates the file if it doesn't exist
cursor = conn.cursor()

for statement in schema_statements:
    cursor.execute(statement)
conn.commit()

print("Empty tables created")

Empty tables created


In [9]:
countries.to_sql("countries", conn, if_exists="append", index=False)
elements.to_sql("elements", conn, if_exists="append", index=False)
flags.to_sql("flags", conn, if_exists="append", index=False)

production_table.to_sql("production", conn, if_exists="append", index=False)
trade_table.to_sql("trade", conn, if_exists="append", index=False)

print("Data loaded into data/project.db")

Data loaded into data/project.db


---
## 7. Verify

Row counts per table, and one join that returns the rows you expect. Then open `data/project.db` in DB Browser and look at it.

In [10]:
# Dangling foreign keys: rows whose area_code has no match in countries (LEFT JOIN -> NULL)
for table in ["production", "trade"]:
    query = f"""
        SELECT COUNT(*) AS dangling
        FROM {table} AS t
        LEFT JOIN countries AS c ON t.area_code = c.area_code
        WHERE c.area_code IS NULL;
    """
    print(table, "dangling country keys:", pd.read_sql(query, conn)["dangling"][0])

production dangling country keys: 0
trade dangling country keys: 0


In [11]:
query = """
SELECT c.area_name, p.year, e.element_name, p.value, e.unit
FROM production p
JOIN countries c ON p.area_code = c.area_code
JOIN elements  e ON p.element_code = e.element_code
WHERE c.area_name = 'Brazil' AND p.year = 2024;
"""
pd.read_sql(query, conn)

,area_name,year,element_name,value,unit
0,Brazil,2024,Area harvested,1943977.0,ha
1,Brazil,2024,Yield,1742.7,kg/ha
2,Brazil,2024,Production,3387724.0,t
